# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umarfarukh786/FlyRank-task1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1: the learned ranking improved the review queue

The public project summary reports that the learned model's Precision@50 was about `0.74`, compared with `0.24` for the hand-written baseline, or roughly a threefold lift. That is a useful decision-support finding because it measures how many of the first 50 reviewed pages matched the observed decline label.

**Methodology question:** where exactly does the label come from, and does it represent a future outcome? In this starter workflow, `is_declining_label` is derived from `trend_direction`, which is itself derived from the current-window trend percentage. I would ask whether the paper's label is defined strictly after the feature window, and whether the reported lift still holds for a future-looking label rather than this proxy. This is a request for clearer temporal alignment, not a challenge to the observed result.

### Finding 2: the model was evaluated with client-holdout validation

The public summary reports a client-holdout split, so pages from a held-out client were not mixed into training. That is a meaningful safeguard because pages from the same client can share hidden patterns, and it makes the claim closer to transfer across clients than a random row split would be.

**Methodology question:** does one 20% client holdout support the full generalization claim? I would ask whether the result is stable across several client groups or folds, whether the held-out clients resemble the intended deployment population, and whether a time-aware test is also needed for a trend-like decision. The reported split supports an observed held-out-client result; it does not by itself establish future-period performance or causal impact on refresh outcomes.

In [1]:
paper_findings = {
    "queue_lift": "Public summary: Precision@50 approximately 0.24 for the rules and 0.74 for the learned model.",
    "client_holdout": "Public summary: validation used a client holdout so whole clients stayed out of training.",
}
for name, finding in paper_findings.items():
    print(f"{name}: {finding}")
print("Questions are recorded in the markdown cell above; this cell preserves the findings as auditable text.")

queue_lift: Public summary: Precision@50 approximately 0.24 for the rules and 0.74 for the learned model.
client_holdout: Public summary: validation used a client holdout so whole clients stayed out of training.
Questions are recorded in the markdown cell above; this cell preserves the findings as auditable text.


## 2. My model under an honest split (before/after)

The “before” result is a stratified random row holdout, included to show why the split choice matters. The “after” result is a client holdout: all rows for selected clients are reserved for testing. Both evaluations use the same prepared rows, same features, same Random Forest settings, same random seed, and the same ranking metrics. The gap is descriptive evidence about validation sensitivity, not proof that either split estimates future business impact.

In [2]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
sys.path.insert(0, str(ROOT / "scripts"))
from ml_utils import MODEL_CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES, precision_at_k

FEATURE_PATH = ROOT / "data" / "processed" / "refresh_feature_vector.csv"
frame = pd.read_csv(FEATURE_PATH)
if frame.empty:
    raise ValueError("The prepared feature vector is empty. Run the preparation pipeline first.")

numeric_features = [column for column in MODEL_NUMERIC_FEATURES if column in frame.columns]
categorical_features = [column for column in MODEL_CATEGORICAL_FEATURES if column in frame.columns]
numeric_frame = frame[numeric_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
categorical_frame = frame[categorical_features].fillna("unknown").astype(str)
encoded_frame = pd.get_dummies(categorical_frame, prefix=categorical_features, dtype=float)
feature_frame = pd.concat([numeric_frame.reset_index(drop=True), encoded_frame.reset_index(drop=True)], axis=1)
target = frame["is_declining_label"].astype(int)

RANDOM_STATE = 42
model_settings = {
    "class_weight": "balanced_subsample",
    "max_depth": 10,
    "min_samples_leaf": 25,
    "n_estimators": 200,
    "n_jobs": -1,
    "random_state": RANDOM_STATE,
}

def evaluate_split(split_name, train_indices, test_indices):
    model = RandomForestClassifier(**model_settings)
    model.fit(feature_frame.iloc[train_indices], target.iloc[train_indices])
    scores = model.predict_proba(feature_frame.iloc[test_indices])[:, 1]
    test_target = target.iloc[test_indices]
    return {
        "split": split_name,
        "train_rows": len(train_indices),
        "test_rows": len(test_indices),
        "test_clients": frame.iloc[test_indices]["client_id"].nunique(),
        "test_base_rate": test_target.mean(),
        "roc_auc": roc_auc_score(test_target, scores),
        "average_precision": average_precision_score(test_target, scores),
        "precision_at_20": precision_at_k(test_target, scores, 20),
        "precision_at_50": precision_at_k(test_target, scores, 50),
        "precision_at_100": precision_at_k(test_target, scores, 100),
        "model": model,
        "scores": scores,
        "test_indices": np.asarray(test_indices),
    }

all_indices = np.arange(len(frame))
random_train, random_test = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=target,
)
random_result = evaluate_split("random_row_holdout_before", random_train, random_test)

clients = frame["client_id"].fillna("unknown").astype(str)
unique_clients = clients.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
test_clients = set(rng.permutation(unique_clients)[:max(1, int(round(len(unique_clients) * 0.20)))])
client_test_mask = clients.isin(test_clients).to_numpy()
client_train = all_indices[~client_test_mask]
client_test = all_indices[client_test_mask]
if target.iloc[client_train].nunique() < 2 or target.iloc[client_test].nunique() < 2:
    raise ValueError("The deterministic client holdout does not contain both label classes.")
client_result = evaluate_split("client_holdout_after", client_train, client_test)

comparison = pd.DataFrame([
    {key: value for key, value in random_result.items() if key not in {"model", "scores", "test_indices"}},
    {key: value for key, value in client_result.items() if key not in {"model", "scores", "test_indices"}},
]).set_index("split")
comparison["precision_at_50_gap_vs_random"] = comparison["precision_at_50"] - comparison.loc["random_row_holdout_before", "precision_at_50"]
print(f"Eligible rows: {len(frame):,} | clients: {frame['client_id'].nunique():,} | overall base rate: {target.mean():.3f}")
display(comparison.round(3))

client_errors = frame.iloc[client_result["test_indices"]][
    ["is_declining_label", "impressions_90d", "sessions_90d", "content_age_days", "avg_position", "ctr"]
].copy()
client_errors["model_score"] = client_result["scores"]
client_errors["predicted_label"] = (client_errors["model_score"] >= 0.5).astype(int)
client_errors["error_type"] = np.select(
    [
        (client_errors["is_declining_label"] == 1) & (client_errors["predicted_label"] == 0),
        (client_errors["is_declining_label"] == 0) & (client_errors["predicted_label"] == 1),
    ],
    ["false_negative", "false_positive"],
    default="correct",
)
print("Three held-out failure examples:")
display(
    client_errors[client_errors["error_type"] != "correct"]
    .sort_values("model_score", ascending=False)
    .head(3)
)
print("Failure interpretation: these rows are measured mistakes under the proxy label; they are not evidence that the model caused decline or recovery.")

Eligible rows: 30,000 | clients: 32 | overall base rate: 0.542


,train_rows,test_rows,test_clients,test_base_rate,roc_auc,average_precision,precision_at_20,precision_at_50,precision_at_100,precision_at_50_gap_vs_random
split,,,,,,,,,,
random_row_holdout_before,24000,6000,31,0.542,0.758,0.769,0.9,0.90,0.9,0.00
client_holdout_after,27675,2325,6,0.391,0.747,0.610,0.7,0.68,0.7,-0.22


Three held-out failure examples:


,is_declining_label,impressions_90d,sessions_90d,content_age_days,avg_position,ctr,model_score,predicted_label,error_type
23250,0,5091,30,144,14.1,0.20,0.737431,1,false_positive
23559,0,1076,9,125,25.6,0.09,0.735212,1,false_positive
23750,0,369,2,112,21.8,0.00,0.733797,1,false_positive


Failure interpretation: these rows are measured mistakes under the proxy label; they are not evidence that the model caused decline or recovery.


## 3. Leakage audit

The audit checks three risks: label-derived columns, decision-derived product outputs, and feature windows that overlap the label. The starter label is a current-window proxy, not a clean future outcome, so the temporal result must be reported as a limitation even though `trend_direction` and `trend_pct` are excluded from the model feature lists.

In [3]:
forbidden_columns = {
    "content_id", "client_id", "trend_direction", "trend_pct", "is_declining_label",
    "health_score", "priority_score", "action_type", "refresh_tier",
}
used_features = set(numeric_features + categorical_features)
feature_audit = pd.DataFrame([
    {
        "check": "label-derived columns excluded",
        "status": "passed" if not used_features.intersection({"trend_direction", "trend_pct", "is_declining_label"}) else "failed",
        "evidence": sorted(used_features.intersection({"trend_direction", "trend_pct", "is_declining_label"})) or "none",
    },
    {
        "check": "IDs excluded from features",
        "status": "passed" if not used_features.intersection({"content_id", "client_id"}) else "failed",
        "evidence": sorted(used_features.intersection({"content_id", "client_id"})) or "none",
    },
    {
        "check": "product decision outputs excluded",
        "status": "passed" if not used_features.intersection({"health_score", "priority_score", "action_type", "refresh_tier"}) else "failed",
        "evidence": sorted(used_features.intersection({"health_score", "priority_score", "action_type", "refresh_tier"})) or "none",
    },
    {
        "check": "future-window safety",
        "status": "limitation",
        "evidence": "The starter label comes from the current trend window; a future-looking label requires a time-aware feature rebuild.",
    },
])
display(feature_audit)
assert not used_features.intersection(forbidden_columns)
print("Leakage conclusion: explicit label/product leakage checks passed; temporal alignment remains a limitation of this proxy-label dataset.")

,check,status,evidence
0,label-derived columns excluded,passed,none
1,IDs excluded from features,passed,none
2,product decision outputs excluded,passed,none
3,future-window safety,limitation,The starter label comes from the current trend...


Leakage conclusion: explicit label/product leakage checks passed; temporal alignment remains a limitation of this proxy-label dataset.


## 4. Claim rewrite

The strongest Week-5 wording was too broad: “the random forest predicts which pages need refreshing.” The evidence supports a narrower statement. It is a ranking result on an anonymized starter slice, against a proxy label, under one client holdout, and it does not measure whether a refresh improves performance.

In [4]:
claim_rewrite = pd.DataFrame([
    {
        "version": "too broad",
        "claim": "The random forest predicts which pages need refreshing and will improve performance.",
    },
    {
        "version": "supported",
        "claim": "On the anonymized starter slice, the random forest measured higher out-of-sample ranking performance than the rule baseline for the observed decline proxy under the selected client holdout.",
    },
    {
        "version": "decision-support",
        "claim": "The score is directional decision-support for prioritizing manual review; it is not a causal estimate of refresh impact or a guarantee about future traffic.",
    },
])
display(claim_rewrite)
print("Safe language used: observed, measured, directional, and decision-support.")

,version,claim
0,too broad,The random forest predicts which pages need re...
1,supported,"On the anonymized starter slice, the random fo..."
2,decision-support,The score is directional decision-support for ...


Safe language used: observed, measured, directional, and decision-support.


## Self-check

Before submitting, rerun the notebook manually and confirm:

- [ ] Two paper findings are named, with a constructive label question and validation question for each.
- [ ] Random-row and client-holdout results use the same model, features, seed, and ranking metrics.
- [ ] The test base rate is shown beside every split comparison.
- [ ] At least three held-out failure examples are displayed without private names, URLs, or queries.
- [ ] Label-derived columns, IDs, and product decision outputs are excluded from features.
- [ ] The current-window proxy-label limitation is stated; no future-performance claim is made.
- [ ] Claims use observed, measured, directional, or decision-support language.
- [ ] The notebook runs top to bottom with no errors.
- [ ] The completed notebook is committed under `work/notebooks/w06_validation_audit.ipynb`.